<a href="https://colab.research.google.com/github/HieuStudyingCS/Learning-AI-Journey/blob/main/Gradient_descent_vs_Momentum_GD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from numpy import linalg
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import random
import time

## Collect Data
In this section, we will test performance between two optimization algorithms Gradient Descent and Momentum-based Gradient Descent

(Sau này có thể phát triển dự án này hơn bằng cách để lại notes về lý thuyết và cách hiểu về 2 thuật toán, có thể benchmark thêm thuật Nesterov's Gradient Descent)

In [ ]:
# Collect data
np.random.seed(42)

In [ ]:
true_weight = 4.5
true_bias = 2.0
nb_samples = 1000
nb_features = 1

In [ ]:
X = 2 * np.random.rand(nb_samples, nb_features)
X

In [ ]:
y = true_weight * X + true_bias + np.random.randn(1000, 1) #  y = 4.5 * X + 2 + noise
y

In [ ]:
plt.scatter(X, y, color='blue', alpha=0.5, s=15, label='Dữ liệu thực tế')
plt.plot(X, true_weight * X + true_bias, color='red', label='Đường thẳng mong muốn')
plt.legend()
plt.show()

## Gradient Descent
### Đạo hàm của hàm một biến
- Gọi $x^{*}$ là điểm local minimum của hàm số, tức là $f'(x^{*}) = 0$.

- Ta thấy một đặc điểm đặc biệt là khi hàm số là đồng biến tại thời điểm $x_t$ (hay $f'(x_t) > 0$) thì $x_t$ luôn nằm về phía bên phải của điểm tối ưu $x^{*}$ và ngược lại
$$f'(x_t) > 0 = f'(x^{*}) \Rightarrow x_t > x^{*}$$   


Một cách tổng quát, ta giả sử $x_t$ nằm về phía phải của $x^{*}$ (hay $f'(x_t) > 0$)

- Nếu mục tiêu của ta là minimize loss function $\mathcal{L}(\mathbf{w})$ vậy thì để điểm tiếp theo $x_{t+1}$ gần với $x^{*}$ hơn thì ta cần di chuyển $x_t$ về phía bên trái (về phía âm) một lượng $\Delta$. Vậy ta có,
$$x_{t+1} = x_t + \Delta \tag{1}$$

Nhận xét:
- Vì hàm số đang là đồng biến, mà ta lại cần $x_t$ di chuyển về phía bên trái (tức là theo chiều âm) vậy thì xét về hướng, $\Delta$ là ngược hướng với đạo hàm tại $x_t$
- Nhận thấy, khi $x_t$ càng xa điểm tối ưu $x^{*}$ về phía bên phải thì để $x_{t+1}$ gần với $x^{*}$ thì $\Delta$ phải càng lớn. Hơn nữa, nếu $x_t$ càng xa $x^{*}$ về phía bên phải thì khi đó đạo hàm tại điểm $x_t$ cũng càng lớn.

Từ 2 nhận xét trên ta rút ra kết luận: $\Delta$ là một đại lượng ngược hướng và có độ lớn tỉ lệ thuận với độ lớn của đạo hàm tại điểm $x_t$.

$$\Delta = - f'(x_t) \tag{2}$$


Từ (1) và (2), ta rút ra công thức cập nhật $x_{t+1}$:
$$x_{t+1} = x_t - \eta f'(x_t)$$

Trong đó, $\eta$ là một số dương được gọi là **Learning Rate** để điều chỉnh độ lớn của đạo hàm trước khi cập nhật, tránh việc cập nhật quá đà sẽ nhảy vọt qua khỏi điểm tối ưu $x^{*}$

Ta cũng để ý một điều nữa đó là khi cập nhật $x_{t+1}$ thì ta hoàn toàn đi ngược hướng với đạo hàm, do vậy mới có cái tên Gradient Descent

### Đạo hàm cho hàm nhiều biến
Một cách tương tự ta thu được công thức cập nhật:
$$\theta_{t+1} = \theta_t - \eta \nabla_{\theta}f(\theta_t)$$

In [ ]:
def graient(w_0, w_1, x):
  return w_0

In [ ]:
def loss_function(y_pred, y):
  return np.mean((y_pred  - y)**2)

In [ ]:
def gradient_descent(X, y, eta=0.01):
  w = np.zeros((X.shape[1], 1))
  loss_history = []
  N = X.shape[0]
  w_0 = 0
  nb_iterations = 0
  bias_history = [float(w_0)]
  w_history = [float(w[0][0])]

  for _ in range(100000):
    nb_iterations += 1
    y_pred = np.dot(X, w) + w_0
    loss = loss_function(y_pred, y)
    loss_history.append(loss)
    dw = (2 / N) * np.dot(X.T, (y_pred - y))
    db = (2 / N) * np.sum(y_pred - y)

    if(loss < 1e-4): break
    if len(loss_history) > 1 and abs(loss_history[-1] - loss_history[-2]) < 1e-9: break

    w -= eta*dw
    w_0 -= eta*db
    bias_history.append(float(w_0))
    w_history.append(float(w[0][0]))

  return w, w_0, loss_history, nb_iterations, w_history, bias_history

In [ ]:
w_gd, b_gd, loss_gd, nb_iterations_GD, w_history_GD, bias_history_GD = gradient_descent(X, y, eta=0.01)
print(f"Trọng số thực tế: w={4.5}, b={2.0}")
print(f"GD tìm được: w={w_gd[0][0]:.3f}, b={b_gd:.3f}")
print(f'Số lượng bước GD: {nb_iterations_GD}')

## Momentum-based Gradient Descent


In [ ]:
def momentum_gradient_descent(X, y, gamma=0.9, eta=0.01):
    w = np.zeros((X.shape[1], 1))
    loss_history = []
    N = X.shape[0]
    w_0 = 0
    nb_iterations = 0

    v_w = np.zeros((X.shape[1], 1))
    v_w0 = 0
    bias_history = [float(w_0)]
    w_history = [float(w[0][0])]

    for _ in range(100000):
        nb_iterations += 1

        y_pred = np.dot(X, w) + w_0
        loss = loss_function(y_pred, y)
        loss_history.append(loss)

        dw = (2 / N) * np.dot(X.T, (y_pred - y))
        db = (2 / N) * np.sum(y_pred - y)

        if (loss < 1e-4):
            break
        if len(loss_history) > 1 and abs(loss_history[-1] - loss_history[-2]) < 1e-9: break

        v_w = gamma * v_w + eta * dw
        v_w0 = gamma * v_w0 + eta * db

        w -= v_w
        w_0 -= v_w0
        bias_history.append(float(w_0))
        w_history.append(float(w[0][0]))

    return w, w_0, loss_history, nb_iterations, w_history, bias_history

## So sánh tổng số bước trước khi hội tụ của GD và MGD

In [ ]:
# Gamma = 0.9
w_gd, b_gd, loss_gd, nb_iterations_GD, w_history_GD, bias_history_GD = gradient_descent(X, y, eta=0.01)
w_mom, b_mom, loss_mom, nb_iterations_MGD, w_history_MGD, bias_history_MGD = momentum_gradient_descent(X, y, eta=0.01, gamma=0.9)

plt.plot(loss_gd[:200], label='Gradient Descent', color='blue')
plt.plot(loss_mom[:200], label='Momentum GD', color='red', linestyle='--')

plt.title(f'So sánh tốc độ hội tụ (200 bước đầu) với gamma = 0.9')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

print(f"Trọng số thực tế: w={4.5}, b={2.0}")
print(f"GD tìm được: w={w_gd[0][0]:.3f}, b={b_gd:.3f}")
print(f"Momentum tìm được: w={w_mom[0][0]:.3f}, b={b_mom:.3f}")
print(f'Số lượng bước GD: {nb_iterations_GD}')
print(f'Số lượng bước Momentum GD: {nb_iterations_MGD}')

In [ ]:
# Gamma = 0.7
w_gd, b_gd, loss_gd, nb_iterations_GD, w_history_GD, bias_history_GD = gradient_descent(X, y, eta=0.01)
w_mom, b_mom, loss_mom, nb_iterations_MGD, w_history_MGD, bias_history_MGD = momentum_gradient_descent(X, y, eta=0.01, gamma=0.7)

plt.plot(loss_gd[:200], label='Gradient Descent', color='blue')
plt.plot(loss_mom[:200], label='Momentum GD', color='red', linestyle='--')

plt.title(f'So sánh tốc độ hội tụ (200 bước đầu) với gamma = 0.7')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

print(f"Trọng số thực tế: w={4.5}, b={2.0}")
print(f"GD tìm được: w={w_gd[0][0]:.3f}, b={b_gd:.3f}")
print(f"Momentum tìm được: w={w_mom[0][0]:.3f}, b={b_mom:.3f}")
print(f'Số lượng bước GD: {nb_iterations_GD}')
print(f'Số lượng bước Momentum GD: {nb_iterations_MGD}')

In [ ]:
# Gamma = 0.5
w_gd, b_gd, loss_gd, nb_iterations_GD, w_history_GD, bias_history_GD = gradient_descent(X, y, eta=0.01)
w_mom, b_mom, loss_mom, nb_iterations_MGD, w_history_MGD, bias_history_MGD = momentum_gradient_descent(X, y, eta=0.01, gamma=0.5)

plt.plot(loss_gd[:200], label='Gradient Descent', color='blue')
plt.plot(loss_mom[:200], label='Momentum GD', color='red', linestyle='--')

plt.title(f'So sánh tốc độ hội tụ (200 bước đầu) với gamma = 0.5')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

print(f"Trọng số thực tế: w={4.5}, b={2.0}")
print(f"GD tìm được: w={w_gd[0][0]:.3f}, b={b_gd:.3f}")
print(f"Momentum tìm được: w={w_mom[0][0]:.3f}, b={b_mom:.3f}")
print(f'Số lượng bước GD: {nb_iterations_GD}')
print(f'Số lượng bước Momentum GD: {nb_iterations_MGD}')

## Nesterov accelerated gradient descent

In [ ]:
def nesterov_momentum_gradient_descent(X, y, gamma=0.9, eta=0.01):
  w = np.zeros((X.shape[1], 1))
  N = X.shape[0]
  v_w0, w_0 = 0, 0
  v_w = np.zeros((X.shape[1], 1))
  nb_iterations = 0
  loss_history = []
  bias_history = [float(w_0)]
  w_history = [float(w[0][0])]

  for _ in range(100000):
    nb_iterations += 1

    y_pred = np.dot(X, w) + w_0
    loss = loss_function(y_pred, y)

    loss_history.append(loss)

    if loss < 1e-4: break
    if len(loss_history) > 1 and abs(loss_history[-1] - loss_history[-2]) < 1e-9: break

    w_tmp = w - gamma * v_w
    w0_tmp = w_0 - gamma * v_w0

    y_pred_tmp = X @ w_tmp + w0_tmp
    dw_tmp = (2 / N) * np.dot(X.T, (y_pred_tmp - y))
    db_tmp = (2 / N) * np.sum(y_pred_tmp -y)

    v_w = gamma * v_w + eta * dw_tmp
    v_w0 = gamma * v_w0 + eta * db_tmp

    w -= v_w
    w_0 -= v_w0

    bias_history.append(float(w_0))
    w_history.append(float(w[0][0]))

  return w, w_0, loss_history, nb_iterations, w_history, bias_history

In [ ]:
w_gd, b_gd, loss_gd, iter_gd, w_history_GD, bias_history_GD = gradient_descent(X, y, eta=0.01)
w_mom, b_mom, loss_mom, iter_mom, w_history_MGD, bias_history_MGD = momentum_gradient_descent(X, y, eta=0.01, gamma=0.9)
w_nes, b_nes, loss_nes, iter_nes, w_history_nes, bias_history_nes = nesterov_momentum_gradient_descent(X, y, eta=0.01, gamma=0.9)

plt.plot(loss_gd[:200], label='Gradient Descent', color='blue')
plt.plot(loss_mom[:200], label='Momentum GD', color='red', linestyle='--')
plt.plot(loss_nes[:200], label='Nesterov GD', color='green', linestyle='-.')

plt.title('So sánh tốc độ hội tụ (200 bước đầu) với gamma = 0.9')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

print(f"Trọng số thực tế: w={{4.5}}, b={{2.0}}")
print(f"GD tìm được: w={w_gd[0][0]:.3f}, b={b_gd:.3f}")
print(f"Momentum tìm được: w={w_mom[0][0]:.3f}, b={b_mom:.3f}")
print(f"Nesterov tìm được: w={w_nes[0][0]:.3f}, b={b_nes:.3f}")
print(f"Số lượng bước GD: {iter_gd}")
print(f"Số lượng bước Momentum: {iter_mom}")
print(f"Số lượng bước Nesterov: {iter_nes}")

In [ ]:
w_gd, b_gd, loss_gd, iter_gd, w_history_GD, bias_history_GD = gradient_descent(X, y, eta=0.1)
w_mom, b_mom, loss_mom, iter_mom, w_history_MGD, bias_history_MGD = momentum_gradient_descent(X, y, eta=0.1, gamma=0.7)
w_nes, b_nes, loss_nes, iter_nes, w_history_nes, bias_history_nes = nesterov_momentum_gradient_descent(X, y, eta=0.1, gamma=0.7)

plt.plot(loss_gd[:200], label='Gradient Descent', color='blue')
plt.plot(loss_mom[:200], label='Momentum GD', color='red', linestyle='--')
plt.plot(loss_nes[:200], label='Nesterov GD', color='green', linestyle='-.')

plt.title('So sánh tốc độ hội tụ (200 bước đầu) với gamma = 0.9 và eta = 0.1')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

print(f"Trọng số thực tế: w={{4.5}}, b={{2.0}}")
print(f"GD tìm được: w={w_gd[0][0]:.3f}, b={b_gd:.3f}")
print(f"Momentum tìm được: w={w_mom[0][0]:.3f}, b={b_mom:.3f}")
print(f"Nesterov tìm được: w={w_nes[0][0]:.3f}, b={b_nes:.3f}")
print(f"Số lượng bước GD: {iter_gd}")
print(f"Số lượng bước Momentum: {iter_mom}")
print(f"Số lượng bước Nesterov: {iter_nes}")

In [ ]:
w_gd, b_gd, loss_gd, iter_gd, w_history_GD, bias_history_GD = gradient_descent(X, y, eta=0.02)
w_mom, b_mom, loss_mom, iter_mom, w_history_MGD, bias_history_MGD = momentum_gradient_descent(X, y, eta=0.02, gamma=0.9)
w_nes, b_nes, loss_nes, iter_nes, w_history_nes, bias_history_nes = nesterov_momentum_gradient_descent(X, y, eta=0.02, gamma=0.9)

plt.plot(loss_gd[:200], label='Gradient Descent', color='blue')
plt.plot(loss_mom[:200], label='Momentum GD', color='red', linestyle='--')
plt.plot(loss_nes[:200], label='Nesterov GD', color='green', linestyle='-.')

plt.title('So sánh tốc độ hội tụ (200 bước đầu) với gamma = 0.9 và eta = 0.02')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

print(f"Trọng số thực tế: w={{4.5}}, b={{2.0}}")
print(f"GD tìm được: w={w_gd[0][0]:.3f}, b={b_gd:.3f}")
print(f"Momentum tìm được: w={w_mom[0][0]:.3f}, b={b_mom:.3f}")
print(f"Nesterov tìm được: w={w_nes[0][0]:.3f}, b={b_nes:.3f}")
print(f"Số lượng bước GD: {iter_gd}")
print(f"Số lượng bước Momentum: {iter_mom}")
print(f"Số lượng bước Nesterov: {iter_nes}")

## Contour và Animation cho cả 3 thuật toán

In [ ]:
from IPython.display import HTML, display
import matplotlib as mpl

_, _, _, _, w_hist_gd, b_hist_gd = gradient_descent(X, y, eta=0.1)
_, _, _, _, w_hist_mom, b_hist_mom = momentum_gradient_descent(X, y, eta=0.1, gamma=0.7)
_, _, _, _, w_hist_nes, b_hist_nes = nesterov_momentum_gradient_descent(X, y, eta=0.1, gamma=0.7)

w_vals = np.linspace(-1, 6, 100)
b_vals = np.linspace(-1, 4, 100)
W, B = np.meshgrid(w_vals, b_vals)
Z = np.zeros_like(W)

for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        y_pred_grid = X * W[i, j] + B[i, j]
        Z[i, j] = np.mean((y_pred_grid - y)**2)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contour(W, B, Z, levels=np.logspace(-1, 3, 20), cmap='viridis', alpha=0.5)
ax.plot(4.5, 2.0, 'k*', markersize=10, label='Global Minimum (4.5, 2.0)')

line_gd, = ax.plot([], [], 'b-', label='Gradient Descent', alpha=0.8)
line_mom, = ax.plot([], [], 'r--', label='Momentum GD', alpha=0.8)
line_nes, = ax.plot([], [], 'g-.', label='Nesterov GD', alpha=0.8)

point_gd, = ax.plot([], [], 'bo', markersize=4)
point_mom, = ax.plot([], [], 'ro', markersize=4)
point_nes, = ax.plot([], [], 'go', markersize=4)

ax.set_xlabel('Weight (w)')
ax.set_ylabel('Bias (b)')
ax.set_title('Optimization Trajectory')
ax.legend()

def init():
    line_gd.set_data([], [])
    line_mom.set_data([], [])
    line_nes.set_data([], [])
    point_gd.set_data([], [])
    point_mom.set_data([], [])
    point_nes.set_data([], [])
    return line_gd, line_mom, line_nes, point_gd, point_mom, point_nes

def update(frame):
    if frame < len(w_hist_gd):
        line_gd.set_data(w_hist_gd[:frame+1], b_hist_gd[:frame+1])
        point_gd.set_data([w_hist_gd[frame]], [b_hist_gd[frame]])

    if frame < len(w_hist_mom):
        line_mom.set_data(w_hist_mom[:frame+1], b_hist_mom[:frame+1])
        point_mom.set_data([w_hist_mom[frame]], [b_hist_mom[frame]])

    if frame < len(w_hist_nes):
        line_nes.set_data(w_hist_nes[:frame+1], b_hist_nes[:frame+1])
        point_nes.set_data([w_hist_nes[frame]], [b_hist_nes[frame]])

    ax.set_title(f'Trajectory Comparison - Iteration: {frame}')
    return line_gd, line_mom, line_nes, point_gd, point_mom, point_nes

max_frames = max(len(w_hist_gd), len(w_hist_mom), len(w_hist_nes))
frames_to_show = min(max_frames, 250)

ani = animation.FuncAnimation(fig, update, frames=frames_to_show, init_func=init, blit=True, interval=30)

mpl.rcParams['animation.embed_limit'] = 50.0

html_anim = HTML(ani.to_jshtml())
plt.close(fig)
display(html_anim)